# SISTEMAS DE BIG DATA - Examen 2ª Evaluación

### **Nombre**:

**INSTRUCCIONES**:

- Si realizas este examen desde tu ordenador debes **grabar la pantalla** con OBS Studio en formato MKV y entregar el vídeo junto con el examen. Para entregarlo, debes subirlo a OneDrive y adjuntar fichero de texto con la URL del recurso compartido.
- Si lo haces en un equipo del centro, grabaré yo la pantalla desde el ordenador del profesor.
- Debes contestar cada pregunta del examen en la celda del cuaderno Jupyter que hay después de cada pregunta. Si necesitas más celdas, puedes agregarlas a continuación de la que hay.
- Todo tu código **tiene que estár ejecutado**.
- La **entrega** del examen práctico se realizará en el canal de Teams habilitado a tal efecto y consistirá en:
  - Notebook de Jupyter (`.ipynb`).
  - Notebook exportado en formato Markdown (`.md`)
  - Fichero de texto (`.txt.`) con la URL al vídeo compartido en caso de haberlo hecho en tu ordenador.
  - Estos tres ficheros deberán entregarse en un único fichero comprimido en formato ZIP (`.zip`) con el nombre `{apellidos}, {nombre} - SBD Ev2`

Ejecuta la siguiente celda para generar el archivo `flota_rebelde.txt` con el que trabajás en este examen

In [194]:
%%writefile flota_rebelde.txt
name~!~base_asignada~!~naves_disponibles
CR90 corvette~!~Base Yavin 4~!~15
X-wing~!~Base Echo (Hoth)~!~120
Y-wing~!~Base Echo (Hoth)~!~45 naves
Millennium Falcon~!~Flota Nómada~!~1
A-wing~!~Base Endor~!~Error de sensor
Rebel transport~!~Punto de encuentro~!~Ocho
B-wing~!~Astillero Sullust~!~30
EF76 Nebulon-B escort frigate~!~Flota Nómada~!~4
Calamari Cruiser~!~Órbita Mon Cala~!~Desconocido
Star Destroyer~!~Hangar Secreto (Capturado)~!~2

Overwriting flota_rebelde.txt


## EXAMEN PRÁCTICO: Logística de la flota rebelde

Trabajas en el equipo de logística y suministros de la Alianza Rebelde. Recientemente, habéis recibido un archivo de texto con el inventario actual de naves espaciales disponibles en vuestras bases secretas. Sin embargo, este archivo fue generado por un sistema antiguo y contiene errores de formato.

Tu misión es extraer este inventario, limpiarlo y cruzarlo con el catálogo oficial de naves de la API de Star Wars (SWAPI) para calcular la capacidad de carga total de la flota.



### Apartado A: Ingesta y limpieza de la fuente estática (2.5 puntos)

El sistema legado ha exportado el inventario en el archivo `flota_rebelde.txt`. Al inspeccionarlo, notas que el separador de columnas es una secuencia extraña de caracteres (`~!~`) y que la columna numérica tiene datos corruptos.

1.  Carga el archivo `flota_rebelde.txt` en un DataFrame de Pandas. 
2.  La columna `naves_disponibles` contiene basura textual en algunas filas (ej. "5 naves", "Error"). Lee inicialmente esta columna como texto (cadena).
3.  Limpia la columna `naves_disponibles` forzando su conversión a tipo numérico. Asegúrate de transformar los textos irreconocibles en valores nulos (`NaN`).
4.  Elimina las filas que hayan quedado con valor nulo en dicha columna.

In [195]:
def limpiezaNaves(valor_col):
    valor_col = valor_col.replace("naves","")
    valor_col = valor_col.replace("Ocho","8")
    return valor_col

In [196]:
import pandas as pd
df_ejer1 = pd.read_csv("./flota_rebelde.txt", sep="~!~", engine="python",dtype={"naves_disponibles": "string"}, na_values =["Error de sensor","Desconocido"],converters={"naves_disponibles":limpiezaNaves})
df_ejer1 = df_ejer1.dropna(subset=["naves_disponibles"])

/tmp/ipykernel_242/3329166864.py:2: ParserWarning: Both a converter and dtype were specified for column naves_disponibles - only the converter will be used.
  df_ejer1 = pd.read_csv("./flota_rebelde.txt", sep="~!~", engine="python",dtype={"naves_disponibles": "string"}, na_values =["Error de sensor","Desconocido"],converters={"naves_disponibles":limpiezaNaves})


In [197]:
df_ejer1.head(12)

,name,base_asignada,naves_disponibles
0,CR90 corvette,Base Yavin 4,15
1,X-wing,Base Echo (Hoth),120
2,Y-wing,Base Echo (Hoth),45
3,Millennium Falcon,Flota Nómada,1
5,Rebel transport,Punto de encuentro,8
6,B-wing,Astillero Sullust,30
7,EF76 Nebulon-B escort frigate,Flota Nómada,4
9,Star Destroyer,Hangar Secreto (Capturado),2



### Apartado B: Extracción de datos desde API REST (2.5 puntos)

Necesitamos obtener las especificaciones técnicas oficiales de todas las naves del universo Star Wars.

1.  Utilizando la librería `requests`, realiza peticiones `GET` al endpoint oficial de naves: `https://swapi.dev/api/starships/`.
2.  Carga la lista completa de naves en un único DataFrame de Pandas llamado `df_catalogo`.

*(Nota: Si no consigues hacer funcionar la petición a la API o la paginación, carga el archivo `swapi_starships_simulado.csv` en `df_catalogo` y continúa con el siguiente apartado).*

In [198]:
import requests

response = requests.get("https://swapi.dev/api/starships/")
if response.status_code == 200:
    datos = response.json()
    print("Conexion_Establecida")
else:
     print(f"Error. {response.status_code}")


Conexion_Establecida


In [199]:
datos["next"]

'https://swapi.dev/api/starships/?page=2'

In [200]:
import pandas as pd

df_catalogo = pd.json_normalize(datos["results"])
while datos.get("next"):
    url = datos["next"]
    response = requests.get(url)
    if response.status_code == 200:
        datos = response.json()
        df_1 = pd.json_normalize(datos["results"])
        df_catalogo = pd.concat([df_catalogo, df1], ignore_index=True)


In [201]:
df_catalogo.head(1)

,name,model,manufacturer,cost_in_credits,length,max_atmosphering_speed,crew,passengers,cargo_capacity,consumables,hyperdrive_rating,MGLT,starship_class,pilots,films,created,edited,url
0,CR90 corvette,CR90 corvette,Corellian Engineering Corporation,3500000,150,950,30-165,600,3000000,1 year,2.0,60,corvette,[],"[https://swapi.dev/api/films/1/, https://swapi...",2014-12-10T14:20:33.369000Z,2014-12-20T21:23:49.867000Z,https://swapi.dev/api/starships/2/


### Apartado C: Transformación y cruce de datos (2.5 puntos)

Ahora debes unificar la información local con la oficial y hacer los cálculos logísticos.

1.  Realiza un cruce entre tu DataFrame del inventario limpio (Apartado A) y el DataFrame del catálogo oficial (Apartado B), utilizando el nombre de la nave como clave de unión.
2.  Crea una nueva columna calculada llamada `capacidad_total_flota`. Esta debe ser el resultado de multiplicar las `naves_disponibles` (de tu inventario) por la carga especificada en la API oficial.

In [202]:
df_union = df_catalogo.merge(df_ejer1, on="name",how="left")

In [203]:
df_union.head(10)

df_union = df_union.fillna(0)

In [204]:
df_union.head(3)

,name,model,manufacturer,cost_in_credits,length,max_atmosphering_speed,crew,passengers,cargo_capacity,consumables,hyperdrive_rating,MGLT,starship_class,pilots,films,created,edited,url,base_asignada,naves_disponibles
0,CR90 corvette,CR90 corvette,Corellian Engineering Corporation,3500000,150,950,30-165,600,3000000,1 year,2.0,60,corvette,[],"[https://swapi.dev/api/films/1/, https://swapi...",2014-12-10T14:20:33.369000Z,2014-12-20T21:23:49.867000Z,https://swapi.dev/api/starships/2/,Base Yavin 4,15
1,Star Destroyer,Imperial I-class Star Destroyer,Kuat Drive Yards,150000000,"1,600",975,"47,060",n/a,36000000,2 years,2.0,60,Star Destroyer,[],"[https://swapi.dev/api/films/1/, https://swapi...",2014-12-10T15:08:19.848000Z,2014-12-20T21:23:49.870000Z,https://swapi.dev/api/starships/3/,Hangar Secreto (Capturado),2
2,Sentinel-class landing craft,Sentinel-class landing craft,"Sienar Fleet Systems, Cyngus Spaceworks",240000,38,1000,5,75,180000,1 month,1.0,70,landing craft,[],[https://swapi.dev/api/films/1/],2014-12-10T15:48:00.586000Z,2014-12-20T21:23:49.873000Z,https://swapi.dev/api/starships/5/,0,0


In [205]:
def limpieza2(valor):
    valor = valor.replace("unknown","0")
    return valor


In [206]:
df_union["cargo_capacity"] = df_union["cargo_capacity"].apply(limpieza2)

In [207]:
df_union["capacidad_total_flota"] = df_union["naves_disponibles"].astype("int32") * df_union["cargo_capacity"].astype("int32")

In [208]:
df_union.head()

,name,model,manufacturer,cost_in_credits,length,max_atmosphering_speed,crew,passengers,cargo_capacity,consumables,...,MGLT,starship_class,pilots,films,created,edited,url,base_asignada,naves_disponibles,capacidad_total_flota
0,CR90 corvette,CR90 corvette,Corellian Engineering Corporation,3500000,150,950,30-165,600,3000000,1 year,...,60,corvette,[],"[https://swapi.dev/api/films/1/, https://swapi...",2014-12-10T14:20:33.369000Z,2014-12-20T21:23:49.867000Z,https://swapi.dev/api/starships/2/,Base Yavin 4,15,45000000
1,Star Destroyer,Imperial I-class Star Destroyer,Kuat Drive Yards,150000000,"1,600",975,"47,060",n/a,36000000,2 years,...,60,Star Destroyer,[],"[https://swapi.dev/api/films/1/, https://swapi...",2014-12-10T15:08:19.848000Z,2014-12-20T21:23:49.870000Z,https://swapi.dev/api/starships/3/,Hangar Secreto (Capturado),2,72000000
2,Sentinel-class landing craft,Sentinel-class landing craft,"Sienar Fleet Systems, Cyngus Spaceworks",240000,38,1000,5,75,180000,1 month,...,70,landing craft,[],[https://swapi.dev/api/films/1/],2014-12-10T15:48:00.586000Z,2014-12-20T21:23:49.873000Z,https://swapi.dev/api/starships/5/,0,0,0
3,Death Star,DS-1 Orbital Battle Station,"Imperial Department of Military Research, Sien...",1000000000000,120000,n/a,"342,953","843,342",1000000000000,3 years,...,10,Deep Space Mobile Battlestation,[],[https://swapi.dev/api/films/1/],2014-12-10T16:36:50.509000Z,2014-12-20T21:26:24.783000Z,https://swapi.dev/api/starships/9/,0,0,0
4,Millennium Falcon,YT-1300 light freighter,Corellian Engineering Corporation,100000,34.37,1050,4,6,100000,2 months,...,75,Light freighter,"[https://swapi.dev/api/people/13/, https://swa...","[https://swapi.dev/api/films/1/, https://swapi...",2014-12-10T16:59:45.094000Z,2014-12-20T21:23:49.880000Z,https://swapi.dev/api/starships/10/,Flota Nómada,1,100000


### Apartado D: Almacenamiento y salida (2.5 puntos)

El sistema de Inteligencia de Negocio y el Data Lake requieren que exportes los resultados en dos formatos distintos.

1.  **Capa de consumo (analistas):** exporta el DataFrame final resultante del Apartado C a un archivo CSV llamado `reporte_logistico.csv`. Debes asegurarte de utilizar coma (`,`) como separador y excluir explícitamente el índice numérico de Pandas.
2.  **Capa Plata (Data Lake):** para almacenar el histórico de forma eficiente para la CPU y en almacenamiento en frío, exporta el mismo DataFrame a formato **Apache Parquet**. Llama al archivo `historico_flota.parquet` y aplica compresión `gzip`.

In [184]:
df_union.to_csv("reporte_logistico.csv",sep=",",index= False)

In [214]:
df_union["base_asignada"] = df_union["base_asignada"].astype("string")
df_union["naves_disponibles"] = df_union["naves_disponibles"].astype("string")

In [215]:
df_union.to_parquet("historico_flota.parquet", compression="gzip")

In [191]:
df_union ["capacidad_total_flota"] = df_union["capacidad_total_flota"].astype("object")